# 초보자용 테이블 생성/데이터 입력 노트북

이 노트북은 다음 순서로 사용합니다.
1. `TABLE_NAME`, `SCHEMA`를 직접 작성
2. 테이블 생성 셀 실행
3. `ROWS`에 데이터 작성
4. 데이터 입력/조회 셀 실행


# STS 문항 템플릿 데이터

In [2]:
# 1) 테이블 이름 + 스키마 설정
from pathlib import Path
import re
import sqlite3

# DB 경로: 프로젝트 루트 app.db (필요 시 이 줄만 수정)
# DB_PATH = Path(r"C:\Users\user\workspace\2.0-modular\app.db")
DB_PATH = Path(r"C:\Users\kmj\project\__project_original\2.0-modular\app.db")
TABLE_NAME = 'parent_item_choice'  # 원하는 테이블 이름으로 변경

# type: INTEGER(정수) / REAL(실수) / TEXT / BLOB(바이너리) 중 하나 사용
# constraints 예시: 'PRIMARY KEY AUTOINCREMENT', 'NOT NULL', 'DEFAULT 0'
SCHEMA = [
    {'name': 'test_id', 'type': 'TEXT', 'constraints': 'PRIMARY KEY'},
    {'name': 'sub_test_json', 'type': 'TEXT', 'constraints': 'NOT NULL'},
    {'name': 'item_template', 'type': 'TEXT', 'constraints': 'NOT NULL'},
    {'name': 'meta_json', 'type': 'TEXT', 'constraints': 'NOT NULL'}
]

VALID_TYPES = {'INTEGER', 'REAL', 'TEXT', 'BLOB'}
identifier_re = re.compile(r'^[A-Za-z_][A-Za-z0-9_]*$')

def validate_identifier(name: str) -> None:
    if not identifier_re.match(name):
        raise ValueError(f'잘못된 이름: {name} (영문/숫자/언더스코어만 허용, 숫자로 시작 불가)')

validate_identifier(TABLE_NAME)
seen = set()
for col in SCHEMA:
    col_name = col['name']
    col_type = col['type'].upper()
    validate_identifier(col_name)
    if col_name in seen:
        raise ValueError(f'중복 컬럼명: {col_name}')
    if col_type not in VALID_TYPES:
        raise ValueError(f'지원하지 않는 타입: {col_type}')
    seen.add(col_name)

print('설정 확인 완료')
print('DB_PATH =', DB_PATH)
print('TABLE_NAME =', TABLE_NAME)
print('SCHEMA =', SCHEMA)

설정 확인 완료
DB_PATH = C:\Users\kmj\project\__project_original\2.0-modular\app.db
TABLE_NAME = parent_item_choice
SCHEMA = [{'name': 'test_id', 'type': 'TEXT', 'constraints': 'PRIMARY KEY'}, {'name': 'sub_test_json', 'type': 'TEXT', 'constraints': 'NOT NULL'}, {'name': 'item_template', 'type': 'TEXT', 'constraints': 'NOT NULL'}, {'name': 'meta_json', 'type': 'TEXT', 'constraints': 'NOT NULL'}]


In [3]:
# 2) 테이블 생성
column_defs = []
for col in SCHEMA:
    c_name = col['name']
    c_type = col['type'].upper()
    c_constraints = col.get('constraints', '').strip()
    part = f'"{c_name}" {c_type}'
    if c_constraints:
        part += f' {c_constraints}'
    column_defs.append(part)

create_sql = f'CREATE TABLE IF NOT EXISTS "{TABLE_NAME}" (' + ', '.join(column_defs) + ')'
print('실행 SQL:', create_sql)

conn = sqlite3.connect(DB_PATH)
try:
    conn.execute(create_sql)
    conn.commit()
    print(f'테이블 생성 완료: {TABLE_NAME}')
finally:
    conn.close()

실행 SQL: CREATE TABLE IF NOT EXISTS "parent_item_choice" ("test_id" TEXT PRIMARY KEY, "sub_test_json" TEXT NOT NULL, "item_template" TEXT NOT NULL, "meta_json" TEXT NOT NULL)
테이블 생성 완료: parent_item_choice


## 영아용

In [ ]:
# 3) 스키마 기준 데이터 입력
import json

# 자동 증가 PK 컬럼은 입력 대상에서 자동 제외됩니다.
insert_columns = []
schema_type_map = {}
for col in SCHEMA:
    c_name = col['name']
    c_type = col['type'].upper()
    c_constraints = col.get('constraints', '').upper()
    schema_type_map[c_name] = c_type
    if 'PRIMARY KEY' in c_constraints and 'AUTOINCREMENT' in c_constraints:
        continue
    insert_columns.append(c_name)

print('입력 대상 컬럼:', insert_columns)

# 아래 데이터는 반드시 insert_columns와 같은 키를 가져야 합니다.
# TEXT 컬럼은 dict/list를 넣으면 자동으로 JSON 문자열로 변환합니다.
ROWS = [
    {
        'test_id': 'STS_CO_PG_I',
        'sub_test_json': {
            'age_range': {
                'as_of_time': '00:00:00',
                'start_inclusive': [1, 0, 0],
                'end_exclusive': [3, 0, 0],
            },
            'gender': ['male', 'female'],
        },
        'item_template': { # 반복 생성
            str(i): {
                '1': '전혀 그렇지 않다',
                '2': '드물게 그렇다',
                '3': '때때로 그렇다',
                '4': '자주 그렇다',
                '5': '거의 항상 그렇다'
            }
            for i in range(1, 43) # 총 42문항
        },
        # 'item_template': {
        #     '1': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '2': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '3': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '4': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '5': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '6': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '7': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '8': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '9': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '10': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '11': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '12': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '13': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '14': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '15': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '16': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '17': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '18': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '19': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '20': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '21': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '22': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '23': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '24': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '25': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '26': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '27': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '28': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '29': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '30': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '31': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '32': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '33': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '34': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '35': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '36': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '37': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '38': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '39': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '40': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '41': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'},
        #     '42': {'1': '전혀 그렇지 않다', '2': '드물게 그렇다', '3': '때때로 그렇다', '4': '자주 그렇다', '5': '거의 항상 그렇다'}
        # },
        'meta_json': {'name': 'STS 6요인 기질검사 영아용',
                      'item_count': 42
        },
    },
]

def normalize_value(col_name: str, value):
    col_type = schema_type_map[col_name]
    if col_type == 'TEXT' and isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return value

expected_keys = set(insert_columns)
normalized_rows = []
for i, row in enumerate(ROWS, start=1):
    if set(row.keys()) != expected_keys:
        raise ValueError(
            f'{i}번째 ROW 키 불일치: expected={sorted(expected_keys)}, got={sorted(row.keys())}. '
            'SCHEMA 컬럼명과 ROW 키 이름을 정확히 맞추세요.'
        )
    normalized = {k: normalize_value(k, row[k]) for k in insert_columns}
    normalized_rows.append(normalized)

placeholders = ', '.join(['?'] * len(insert_columns))
col_sql = ', '.join([f'\"{c}\"' for c in insert_columns])
insert_sql = f'INSERT INTO \"{TABLE_NAME}\" ({col_sql}) VALUES ({placeholders})'
values = [tuple(row[c] for c in insert_columns) for row in normalized_rows]

conn = sqlite3.connect(DB_PATH)
try:
    conn.executemany(insert_sql, values)
    conn.commit()
    print(f'입력 완료: {len(values)}건')
finally:
    conn.close()

입력 대상 컬럼: ['test_id', 'sub_test_json', 'item_template', 'meta_json']
입력 완료: 1건


In [5]:
# 4) 결과 조회
conn = sqlite3.connect(DB_PATH)
try:
    cur = conn.execute(f'SELECT * FROM "{TABLE_NAME}" ORDER BY rowid DESC')
    rows = cur.fetchall()
    print(f'총 {len(rows)}건')
    for row in rows:
        print(row)
finally:
    conn.close()

총 1건
('STS_CO_PG_I', '{"age_range": {"as_of_time": "00:00:00", "start_inclusive": [1, 0, 0], "end_exclusive": [3, 0, 0]}, "gender": ["male", "female"]}', '{"1": {"1": "전혀 그렇지 않다", "2": "드물게 그렇다", "3": "때때로 그렇다", "4": "자주 그렇다", "5": "거의 항상 그렇다"}, "2": {"1": "전혀 그렇지 않다", "2": "드물게 그렇다", "3": "때때로 그렇다", "4": "자주 그렇다", "5": "거의 항상 그렇다"}, "3": {"1": "전혀 그렇지 않다", "2": "드물게 그렇다", "3": "때때로 그렇다", "4": "자주 그렇다", "5": "거의 항상 그렇다"}, "4": {"1": "전혀 그렇지 않다", "2": "드물게 그렇다", "3": "때때로 그렇다", "4": "자주 그렇다", "5": "거의 항상 그렇다"}, "5": {"1": "전혀 그렇지 않다", "2": "드물게 그렇다", "3": "때때로 그렇다", "4": "자주 그렇다", "5": "거의 항상 그렇다"}, "6": {"1": "전혀 그렇지 않다", "2": "드물게 그렇다", "3": "때때로 그렇다", "4": "자주 그렇다", "5": "거의 항상 그렇다"}, "7": {"1": "전혀 그렇지 않다", "2": "드물게 그렇다", "3": "때때로 그렇다", "4": "자주 그렇다", "5": "거의 항상 그렇다"}, "8": {"1": "전혀 그렇지 않다", "2": "드물게 그렇다", "3": "때때로 그렇다", "4": "자주 그렇다", "5": "거의 항상 그렇다"}, "9": {"1": "전혀 그렇지 않다", "2": "드물게 그렇다", "3": "때때로 그렇다", "4": "자주 그렇다", "5": "거의 항상 그렇다"}, "10": {"1": "전혀 그렇지 않다", "2": "드물게

## 유아용

In [6]:
# 3) 스키마 기준 데이터 입력
import json

# 자동 증가 PK 컬럼은 입력 대상에서 자동 제외됩니다.
insert_columns = []
schema_type_map = {}
for col in SCHEMA:
    c_name = col['name']
    c_type = col['type'].upper()
    c_constraints = col.get('constraints', '').upper()
    schema_type_map[c_name] = c_type
    if 'PRIMARY KEY' in c_constraints and 'AUTOINCREMENT' in c_constraints:
        continue
    insert_columns.append(c_name)

print('입력 대상 컬럼:', insert_columns)

# 아래 데이터는 반드시 insert_columns와 같은 키를 가져야 합니다.
# TEXT 컬럼은 dict/list를 넣으면 자동으로 JSON 문자열로 변환합니다.
ROWS = [
    {
        'test_id': 'STS_CO_PG_T',
        'sub_test_json': {
            'age_range': {
                'as_of_time': '00:00:00',
                'start_inclusive': [3, 0, 0],
                'end_exclusive': [7, 0, 0],
            },
            'gender': ['male', 'female'],
        },
        'item_template': { # 반복 생성
            str(i): {
                '1': '전혀 그렇지 않다',
                '2': '드물게 그렇다',
                '3': '때때로 그렇다',
                '4': '자주 그렇다',
                '5': '거의 항상 그렇다'
            }
            for i in range(1, 44) # 총 43문항
        },
        'meta_json': {'name': 'STS 6요인 기질검사 유아용',
                      'item_count': 43
        },
    },
]

def normalize_value(col_name: str, value):
    col_type = schema_type_map[col_name]
    if col_type == 'TEXT' and isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return value

expected_keys = set(insert_columns)
normalized_rows = []
for i, row in enumerate(ROWS, start=1):
    if set(row.keys()) != expected_keys:
        raise ValueError(
            f'{i}번째 ROW 키 불일치: expected={sorted(expected_keys)}, got={sorted(row.keys())}. '
            'SCHEMA 컬럼명과 ROW 키 이름을 정확히 맞추세요.'
        )
    normalized = {k: normalize_value(k, row[k]) for k in insert_columns}
    normalized_rows.append(normalized)

placeholders = ', '.join(['?'] * len(insert_columns))
col_sql = ', '.join([f'\"{c}\"' for c in insert_columns])
insert_sql = f'INSERT INTO \"{TABLE_NAME}\" ({col_sql}) VALUES ({placeholders})'
values = [tuple(row[c] for c in insert_columns) for row in normalized_rows]

conn = sqlite3.connect(DB_PATH)
try:
    conn.executemany(insert_sql, values)
    conn.commit()
    print(f'입력 완료: {len(values)}건')
finally:
    conn.close()

입력 대상 컬럼: ['test_id', 'sub_test_json', 'item_template', 'meta_json']
입력 완료: 1건


## 성인용

In [7]:
# 3) 스키마 기준 데이터 입력
import json

# 자동 증가 PK 컬럼은 입력 대상에서 자동 제외됩니다.
insert_columns = []
schema_type_map = {}
for col in SCHEMA:
    c_name = col['name']
    c_type = col['type'].upper()
    c_constraints = col.get('constraints', '').upper()
    schema_type_map[c_name] = c_type
    if 'PRIMARY KEY' in c_constraints and 'AUTOINCREMENT' in c_constraints:
        continue
    insert_columns.append(c_name)

print('입력 대상 컬럼:', insert_columns)

# 아래 데이터는 반드시 insert_columns와 같은 키를 가져야 합니다.
# TEXT 컬럼은 dict/list를 넣으면 자동으로 JSON 문자열로 변환합니다.
ROWS = [
    {
        'test_id': 'STS_CO_SG_Adu',
        'sub_test_json': {
            'age_range': {
                'as_of_time': '00:00:00',
                'start_inclusive': [18, 0, 0],
                'end_exclusive': [100, 0, 0],
            },
            'gender': ['male', 'female'],
        },
        'item_template': { # 반복 생성
            str(i): {
                '1': '전혀 그렇지 않다',
                '2': '드물게 그렇다',
                '3': '때때로 그렇다',
                '4': '자주 그렇다',
                '5': '거의 항상 그렇다'
            }
            for i in range(1, 44) # 총 43문항
        },
        'meta_json': {'name': 'STS 6요인 기질검사 성인용',
                      'item_count': 43
        },
    },
]

def normalize_value(col_name: str, value):
    col_type = schema_type_map[col_name]
    if col_type == 'TEXT' and isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return value

expected_keys = set(insert_columns)
normalized_rows = []
for i, row in enumerate(ROWS, start=1):
    if set(row.keys()) != expected_keys:
        raise ValueError(
            f'{i}번째 ROW 키 불일치: expected={sorted(expected_keys)}, got={sorted(row.keys())}. '
            'SCHEMA 컬럼명과 ROW 키 이름을 정확히 맞추세요.'
        )
    normalized = {k: normalize_value(k, row[k]) for k in insert_columns}
    normalized_rows.append(normalized)

placeholders = ', '.join(['?'] * len(insert_columns))
col_sql = ', '.join([f'\"{c}\"' for c in insert_columns])
insert_sql = f'INSERT INTO \"{TABLE_NAME}\" ({col_sql}) VALUES ({placeholders})'
values = [tuple(row[c] for c in insert_columns) for row in normalized_rows]

conn = sqlite3.connect(DB_PATH)
try:
    conn.executemany(insert_sql, values)
    conn.commit()
    print(f'입력 완료: {len(values)}건')
finally:
    conn.close()

입력 대상 컬럼: ['test_id', 'sub_test_json', 'item_template', 'meta_json']
입력 완료: 1건
